<a href="https://colab.research.google.com/github/GeomaticsCaminosUPM/footprint_attributes/blob/main/examples/example_google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Footprint Attributes package example

Install libraries

In [ ]:
!pip install geopandas
!pip install git+https://github.com/GeomaticsCaminosUPM/footprint_attributes.git
!pip install folium matplotlib mapclassify

  Cloning https://github.com/GeomaticsCaminosUPM/footprint_attributes.git to /tmp/pip-req-build-nmfjqj5o
  Running command git clone --filter=blob:none --quiet https://github.com/GeomaticsCaminosUPM/footprint_attributes.git /tmp/pip-req-build-nmfjqj5o
  Resolved https://github.com/GeomaticsCaminosUPM/footprint_attributes.git to commit 5af5a268888f1be33a479d3dcce78705d9c93446
  Preparing metadata (setup.py) ... done
  Created wheel for footprint_attributes: filename=footprint_attributes-0.1.0-py3-none-any.whl size=12940 sha256=48dc73f64cd3ee972a9a4d1974122c0215d34492c1b656c637a86e102b8cbb52
  Stored in directory: /tmp/pip-ephem-wheel-cache-xkr_nnu9/wheels/77/06/25/387e2cff313ccb053bf44693372eb5c286dbf4af20f521c009
Successfully built footprint_attributes
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
import footprint_attributes
import geopandas as gpd

Documentation of every function can be found using `help()` or under https://github.com/GeomaticsCaminosUPM/footprint_attributes

In [ ]:
help(footprint_attributes.calc_forces)

Help on function calc_forces:

calc_forces(geoms: geopandas.geodataframe.GeoDataFrame, buffer: float = 0, height_column: str = None, min_radius: float = 0) -> geopandas.geodataframe.GeoDataFrame
    Calculates force-based metrics for building footprints based on their geometry and proximity.
    
    Parameters:
        geoms (gpd.GeoDataFrame): A GeoDataFrame containing building footprints as polygon geometries.
        buffer (float): Buffer distance in meters to determine if two buildings are considered touching.
        height_column (str, optional): Column name in `geoms` specifying the building height in meters.
                                       If `None`, all buildings are assumed to have a uniform height of 1.
        min_radius (float, optional): Minimum distance multiplier used to exclude forces that would otherwise 
                                        increase momentum. Forces with a distance below a threshold 
                                        (`min_radius * 

## 1. Load data

Load the footprints geodataframe in `.shp` `.gpkg` `.geojson` format using `geopandas.read_file()`

Download example footprints file

In [ ]:
!wget https://github.com/GeomaticsCaminosUPM/footprint_attributes/raw/refs/heads/main/examples/footprints.gpkg

--2024-12-24 15:06:42--  https://github.com/GeomaticsCaminosUPM/footprint_attributes/raw/refs/heads/main/examples/footprints.gpkg
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/GeomaticsCaminosUPM/footprint_attributes/refs/heads/main/examples/footprints.gpkg [following]
--2024-12-24 15:06:43--  https://raw.githubusercontent.com/GeomaticsCaminosUPM/footprint_attributes/refs/heads/main/examples/footprints.gpkg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 372736 (364K) [application/octet-stream]
Saving to: ‘footprints.gpkg’

footprints.gpkg     100%[===================>] 364.00K  --.-KB/s    in

Load footprints file

In [ ]:
footprints_gdf = gpd.read_file('/content/footprints.gpkg')

- All geometries should be single part and polygons.


  - Multi part geometries (MultiPolygon) would mean two buildings are contained in the same footprint geometry which should not happen.


  - Footprints must be polygons, not linestrings.


Check if all geometries are `Polygon`

In [ ]:
if any(footprints_gdf.geometry.type == 'MultiPolygon'):
    print("There are multiplart geometries.")

if any(footprints_gdf.geometry.type.str.contains('Polygon') == False):
    print("Some geometries are not Polygon")

There are multiplart geometries.


Explode multipart geometries into single parts if needed


Note: The row number will be reset. Maybe you do not know how to match the footprints with your data. An id column is a good idea to solve this issue.

In [ ]:
footprints_gdf = footprints_gdf.explode().reset_index() # The new index column is the row number in the origninal dataframe
footprints_gdf

,index,gid,geometry
0,0,1218.0,"POLYGON ((488600.239 1097577.417, 488591.045 1..."
1,1,1219.0,"POLYGON ((488562.593 1097585.096, 488567.403 1..."
2,2,1220.0,"POLYGON ((488601.331 1097581.438, 488599.351 1..."
3,3,1221.0,"POLYGON ((488517.642 1097593.605, 488517.592 1..."
4,4,1222.0,"POLYGON ((488492.697 1097591.081, 488492.546 1..."
...,...,...,...
939,943,1273.0,"POLYGON ((488958.217 1097631.443, 488961.413 1..."
940,944,1333.0,"POLYGON ((489119.978 1097677.125, 489130.663 1..."
941,945,3632.0,"POLYGON ((489024.946 1097763.628, 489024.157 1..."
942,946,1505.0,"POLYGON ((488924.894 1097824.766, 488928.927 1..."


## 2. Relative position


Determines if the building touches other structures (relative position in the city block). This is done by calculating "forces" that neighboring structures exert on the building. The force is proportional to the contact area (length of touching footprints multiplied by building height) in the normal direction of the touching plane.


Lets calculate the forces with `calc_forces()`

- `buffer`: Buffer distance in meters to determine if two buildings are considered touching.


- `height_column`: Column name in geoms specifying the building height in meters. If None, all buildings are assumed to have a uniform height of 1.


- `min_radius`:   Minimum distance multiplier used to exclude forces that would otherwise increase momentum. Forces with a distance below a threshold (min_radius * equivalent radius) will contribute to the momentum calculation only if they decrease the momentum. The equivalent radius of a building is defined as the radius of a circle with the same area as the building's footprint.

In [ ]:
footprints_gdf = footprint_attributes.calc_forces(footprints_gdf,height_column=None,buffer=0.2,min_radius=0.5) #20 cm of buffer
footprints_gdf

,index,gid,geometry,force,confinement_ratio,angular_acc,angle
0,0,1218.0,"POLYGON ((488600.239 1097577.417, 488591.045 1...",0.697807,2.119091,0.320945,2.465851e+00
1,1,1219.0,"POLYGON ((488562.593 1097585.096, 488567.403 1...",0.641808,0.766662,1.613179,1.159873e+00
2,2,1220.0,"POLYGON ((488601.331 1097581.438, 488599.351 1...",0.341574,7.681600,0.001463,2.060583e-01
3,3,1221.0,"POLYGON ((488517.642 1097593.605, 488517.592 1...",0.020213,179.790671,0.067755,2.266605e+00
4,4,1222.0,"POLYGON ((488492.697 1097591.081, 488492.546 1...",0.287717,0.000000,0.220567,0.000000e+00
...,...,...,...,...,...,...,...
939,943,1273.0,"POLYGON ((488958.217 1097631.443, 488961.413 1...",1.113140,0.000000,0.034521,0.000000e+00
940,944,1333.0,"POLYGON ((489119.978 1097677.125, 489130.663 1...",0.791168,0.000000,0.220128,4.214685e-08
941,945,3632.0,"POLYGON ((489024.946 1097763.628, 489024.157 1...",0.977905,1.111545,1.164580,1.587350e+00
942,946,1505.0,"POLYGON ((488924.894 1097824.766, 488928.927 1...",0.769893,1.679563,0.817435,1.745959e+00


The function returns the input `gpd.GeoDataFrame`, which includes the following new columns:

1. **`angular_acc`**  
   Angular acceleration calculated as: $\text{angular acc} = \frac{\text{momentum} \cdot \text{area}}{\text{inertia}}$
   - **Momentum** is calculated as: $\text{momentum} = \sum (\text{distance} \cdot |\text{force}_i|)$

2. **`force`**  
   Magnitude of the resultant force acting on the footprint, normalized by the square root of the area: $\text{force} = \left| \sum \text{force}_i \right|$

3. **`confinement_ratio`**  
   Proportion of total forces that are confined (counterbalanced by opposing forces): $\text{confinement ratio} = \frac{\sum |\text{force}_i| - \left| \sum \text{force}_i \right|}{\left| \sum \text{force}_i \right|}$

4. **`angle`**  
   Normalized sum of the angles between individual forces and the resultant force: $\text{angle} = \frac{\sum \left( |\text{force}_i| \cdot \text{angle}(\text{force}_i, \sum \text{force}_j) \right)}{\left| \sum \text{force}_i \right|}$


Now determine the relative position using the forces calculated before with `relative_position()`

  - **`min_force`**: Significance threshold for the resultant force. Default: `0.166`. (E.g., for a square building with height 1 and side length 1, if a touching structure covers only 1/6 of one side, the resultant force would be 1/6.)


  - **`min_angle`**: Angle threshold (in radians). Default: $\pi / 4$ (45 degrees).


  - **`min_confinement`**: Threshold for confinement. Default: `1` (indicating equal amounts of confined and resultant forces).


  - **`min_angular_acc`**: Threshold for angular acceleration $\frac{momentum * area}{inertia}$. Default: 2.133 (e.g., for a rectangular building with height 1 and sides of length 1 and 0.5, a touching structure covering 1/3 of two sides in the worst case would have an anuglar acceleration of 2.133)

In [ ]:
footprints_gdf = footprint_attributes.relative_position(footprints_gdf,min_angular_acc=2.133,min_confinement=1,min_angle=0.78,min_force=0.1666)
footprints_gdf

,index,gid,geometry,force,confinement_ratio,angular_acc,angle,relative_position
0,0,1218.0,"POLYGON ((488600.239 1097577.417, 488591.045 1...",0.697807,2.119091,0.320945,2.465851e+00,confined
1,1,1219.0,"POLYGON ((488562.593 1097585.096, 488567.403 1...",0.641808,0.766662,1.613179,1.159873e+00,corner
2,2,1220.0,"POLYGON ((488601.331 1097581.438, 488599.351 1...",0.341574,7.681600,0.001463,2.060583e-01,confined
3,3,1221.0,"POLYGON ((488517.642 1097593.605, 488517.592 1...",0.020213,179.790671,0.067755,2.266605e+00,confined
4,4,1222.0,"POLYGON ((488492.697 1097591.081, 488492.546 1...",0.287717,0.000000,0.220567,0.000000e+00,partial
...,...,...,...,...,...,...,...,...
939,943,1273.0,"POLYGON ((488958.217 1097631.443, 488961.413 1...",1.113140,0.000000,0.034521,0.000000e+00,partial
940,944,1333.0,"POLYGON ((489119.978 1097677.125, 489130.663 1...",0.791168,0.000000,0.220128,4.214685e-08,partial
941,945,3632.0,"POLYGON ((489024.946 1097763.628, 489024.157 1...",0.977905,1.111545,1.164580,1.587350e+00,confined
942,946,1505.0,"POLYGON ((488924.894 1097824.766, 488928.927 1...",0.769893,1.679563,0.817435,1.745959e+00,confined


  Returns the input `gpd.GeoDataFrame` with a new column `"relative_position"`. Classifies buildings into the following categories (priority order):
  1. **"torque"**: Buildings of class **confined** or **corner** with an angular acceleration exceeding the minimum.
  2. **"confined"**: Structures touching on both the left and right lateral sides.
  3. **"corner"**: Structures touching at a corner (determined by force and angle thresholds).
  4. **"partial"**: Structures touching on either the left or right side.
  5. **"isolated"**: No touching structures.

## 3. Irregularity



- **Custom Irregularity Index**: Our own index to quantify the irregularity of building footprints.
  - Formula: $\frac{l \cdot d}{L}$, where:
    - $l$: Length of the shapes outside the convex hull.
    - $d$: Distance of the center of gravity of the shapes outside the hull to the hull.
    - $L$: Total length of the convex hull.

- Function: `calc_shape_irregularity()`


  - **Output**: Returns the same GeoDataFrame with a new column `"shape_irregularity"`.

In [ ]:
footprints_gdf = footprint_attributes.calc_shape_irregularity(footprints_gdf)
footprints_gdf

,index,gid,geometry,force,confinement_ratio,angular_acc,angle,relative_position,shape_irregularity
0,0,1218.0,"POLYGON ((488600.239 1097577.417, 488591.045 1...",0.697807,2.119091,0.320945,2.465851e+00,confined,0.000000
1,1,1219.0,"POLYGON ((488562.593 1097585.096, 488567.403 1...",0.641808,0.766662,1.613179,1.159873e+00,corner,0.000148
2,2,1220.0,"POLYGON ((488601.331 1097581.438, 488599.351 1...",0.341574,7.681600,0.001463,2.060583e-01,confined,0.000000
3,3,1221.0,"POLYGON ((488517.642 1097593.605, 488517.592 1...",0.020213,179.790671,0.067755,2.266605e+00,confined,0.004862
4,4,1222.0,"POLYGON ((488492.697 1097591.081, 488492.546 1...",0.287717,0.000000,0.220567,0.000000e+00,partial,0.463150
...,...,...,...,...,...,...,...,...,...
939,943,1273.0,"POLYGON ((488958.217 1097631.443, 488961.413 1...",1.113140,0.000000,0.034521,0.000000e+00,partial,0.721346
940,944,1333.0,"POLYGON ((489119.978 1097677.125, 489130.663 1...",0.791168,0.000000,0.220128,4.214685e-08,partial,0.000000
941,945,3632.0,"POLYGON ((489024.946 1097763.628, 489024.157 1...",0.977905,1.111545,1.164580,1.587350e+00,confined,0.338223
942,946,1505.0,"POLYGON ((488924.894 1097824.766, 488928.927 1...",0.769893,1.679563,0.817435,1.745959e+00,confined,0.296966


- **Polsby-Popper Index**: A measure of shape compactness (similarity to a circle).
   - Formula: $\text{Polsby-Popper Index} = \frac{4 \pi A}{P^2}$
    where:
      - $A$: Area of the polygon.
      - $P$: Perimeter of the polygon.


- Function: `calc_polsby_popper()`


  - **Output**: Returns the same GeoDataFrame with a new column `"polsby_popper"`.


In [ ]:
footprints_gdf = footprint_attributes.calc_polsby_popper(footprints_gdf)
footprints_gdf

,index,gid,geometry,force,confinement_ratio,angular_acc,angle,relative_position,shape_irregularity,polsby_popper,polsby_popper_hull
0,0,1218.0,"POLYGON ((488600.239 1097577.417, 488591.045 1...",0.697807,2.119091,0.320945,2.465851e+00,confined,0.000000,0.731584,0.731584
1,1,1219.0,"POLYGON ((488562.593 1097585.096, 488567.403 1...",0.641808,0.766662,1.613179,1.159873e+00,corner,0.000148,0.570544,0.570725
2,2,1220.0,"POLYGON ((488601.331 1097581.438, 488599.351 1...",0.341574,7.681600,0.001463,2.060583e-01,confined,0.000000,0.615186,0.615259
3,3,1221.0,"POLYGON ((488517.642 1097593.605, 488517.592 1...",0.020213,179.790671,0.067755,2.266605e+00,confined,0.004862,0.556486,0.557763
4,4,1222.0,"POLYGON ((488492.697 1097591.081, 488492.546 1...",0.287717,0.000000,0.220567,0.000000e+00,partial,0.463150,0.554642,0.724261
...,...,...,...,...,...,...,...,...,...,...,...
939,943,1273.0,"POLYGON ((488958.217 1097631.443, 488961.413 1...",1.113140,0.000000,0.034521,0.000000e+00,partial,0.721346,0.562271,0.867865
940,944,1333.0,"POLYGON ((489119.978 1097677.125, 489130.663 1...",0.791168,0.000000,0.220128,4.214685e-08,partial,0.000000,0.780750,0.780750
941,945,3632.0,"POLYGON ((489024.946 1097763.628, 489024.157 1...",0.977905,1.111545,1.164580,1.587350e+00,confined,0.338223,0.563278,0.772658
942,946,1505.0,"POLYGON ((488924.894 1097824.766, 488928.927 1...",0.769893,1.679563,0.817435,1.745959e+00,confined,0.296966,0.642602,0.821938


- **Inertia Irregularity**: Inertia of a circle with the same area as the polygon geometry divided by the inertia of the polygon.
    - Formula: $\text{intertia irregularity} = \frac{\text{intertia eq circle}}{\text{itertia}}$


- Function: `calc_inertia_irregularity()`

    - **Output**: Returns the same GeoDataFrame with a new column `"inertia_irregularity"`.

In [ ]:
footprints_gdf = footprint_attributes.calc_inertia_irregularity(footprints_gdf)
footprints_gdf

,index,gid,geometry,force,confinement_ratio,angular_acc,angle,relative_position,shape_irregularity,polsby_popper,polsby_popper_hull,inertia_irregularity
0,0,1218.0,"POLYGON ((488600.239 1097577.417, 488591.045 1...",0.697807,2.119091,0.320945,2.465851e+00,confined,0.000000,0.731584,0.731584,0.445876
1,1,1219.0,"POLYGON ((488562.593 1097585.096, 488567.403 1...",0.641808,0.766662,1.613179,1.159873e+00,corner,0.000148,0.570544,0.570725,0.272441
2,2,1220.0,"POLYGON ((488601.331 1097581.438, 488599.351 1...",0.341574,7.681600,0.001463,2.060583e-01,confined,0.000000,0.615186,0.615259,0.293367
3,3,1221.0,"POLYGON ((488517.642 1097593.605, 488517.592 1...",0.020213,179.790671,0.067755,2.266605e+00,confined,0.004862,0.556486,0.557763,0.262464
4,4,1222.0,"POLYGON ((488492.697 1097591.081, 488492.546 1...",0.287717,0.000000,0.220567,0.000000e+00,partial,0.463150,0.554642,0.724261,0.330931
...,...,...,...,...,...,...,...,...,...,...,...,...
939,943,1273.0,"POLYGON ((488958.217 1097631.443, 488961.413 1...",1.113140,0.000000,0.034521,0.000000e+00,partial,0.721346,0.562271,0.867865,0.466330
940,944,1333.0,"POLYGON ((489119.978 1097677.125, 489130.663 1...",0.791168,0.000000,0.220128,4.214685e-08,partial,0.000000,0.780750,0.780750,0.472282
941,945,3632.0,"POLYGON ((489024.946 1097763.628, 489024.157 1...",0.977905,1.111545,1.164580,1.587350e+00,confined,0.338223,0.563278,0.772658,0.386392
942,946,1505.0,"POLYGON ((488924.894 1097824.766, 488928.927 1...",0.769893,1.679563,0.817435,1.745959e+00,confined,0.296966,0.642602,0.821938,0.429483


## 4. Example code to plot results

In [ ]:
# Plot a map (needs pip install folium matplotlib mapclassify)
footprints_gdf.explore(
    column='relative_position',       # Column to base the color on
    cmap='YlOrRd',        # Color map (Yellow to Red)
    legend=True ,           # Add a legend
    #vmin = 1.8,
    #vmax = 2.5               # Set max and min values
)

## 5. Save results

In [ ]:
footprints_gdf.to_file('path/to/save/file.gpkg')